# Preprocessing Pipelines and ColumnTransformer

> Build repeatable preprocessing and modeling workflows that prevent leakage.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Combine numeric imputation/scaling and categorical imputation/encoding.
- Put preprocessing and the estimator in one `Pipeline`.
- Cross-validate the whole pipeline and keep inference transformations consistent.

A scikit-learn `Pipeline` applies each transformation in order and can be cross-validated as a single estimator. `ColumnTransformer` lets different transformations operate on different columns. See the official [Pipeline documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = pd.DataFrame({"age": [22, 35, None, 52, 46, 28, 41, None], "income": [30, 58, 42, 91, None, 35, 74, 50], "plan": ["basic", "plus", "basic", "premium", "plus", None, "premium", "basic"]})
y = [0, 1, 0, 1, 1, 0, 1, 0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)

In [2]:
numeric_features = ["age", "income"]
categorical_features = ["plan"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])
model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test), zero_division=0))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



## Inspect transformed features and safe inference
`handle_unknown="ignore"` prevents prediction from failing when a category appears that was not seen during fitting. It does not mean the new category is well represented; monitor unseen categories in production.

In [3]:
fitted_preprocessor = model.named_steps["preprocess"]
print("Output feature names:", fitted_preprocessor.get_feature_names_out())
new_customers = pd.DataFrame({"age": [33, None], "income": [60, 48], "plan": ["premium", "new-plan"]})
print("Predictions:", model.predict(new_customers))
print("Probabilities:")
print(model.predict_proba(new_customers).round(3))

Output feature names: ['numeric__age' 'numeric__income' 'categorical__plan_basic'
 'categorical__plan_plus' 'categorical__plan_premium']
Predictions: [1 0]
Probabilities:
[[0.387 0.613]
 [0.54  0.46 ]]


## Common pitfalls
- Do not call `fit_transform` on the complete dataset before splitting.
- Set `random_state` for reproducible data splits where shuffling is appropriate.
- Save the full fitted pipeline, not only the final classifier.
- Keep input column names and data types documented for production inference.

## Practice
Add a categorical “region” column with a missing value. Update `ColumnTransformer` and verify that an unseen region does not crash inference.

In [4]:
X2 = X.copy()
X2["region"] = ["north", "south", "east", "west", "north", "south", None, "east"]
print(X2.head())
print("Practice hint: add 'region' to categorical_features before fitting a new pipeline.")

    age  income     plan region
0  22.0    30.0    basic  north
1  35.0    58.0     plus  south
2   NaN    42.0    basic   east
3  52.0    91.0  premium   west
4  46.0     NaN     plus  north
Practice hint: add 'region' to categorical_features before fitting a new pipeline.
